# Basic interaction with Anthropic Claude models

This example demonstrates how to set up an Anthropic Claude model and send a basic user prompt using a Jypiter notebook.

## 1. Setup

To use the Anthropic API, you need to have an API key:

1. Create or log into your [Anthropic account](https://console.anthropic.com)
2. Go to [API Keys page](https://console.anthropic.com/settings/keys)
3. Click "Create API key", give it a name, and copy the key immediately.
4. Make sure your account has billing enabled, since Claude API usage is pay-per-use.

Once you have the API key, you need to export its value as and environment variable called `ANTHROPIC_API_KEY`. If this env is not defined, you will be asked for your API key (it will not be stored in the notebook).

In [ ]:
!pip install anthropic

import os
import time
from getpass import getpass
from anthropic import Anthropic

# Ask for the API key interactively if not defined as env variable
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

client = Anthropic() # ANTHROPIC_API_KEY should be set as an environment variable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 10.5 MB/s eta 0:00:00
Enter your Anthropic API key: ··········


## 2. Function to query the model

This section defines a helper function to send a user prompt to the model.

In [3]:
def query_model(prompt: str,
                model: str = "claude-3-haiku-20240307",
                max_tokens: int = 2048,
                temperature: float = 0,
                thinking_budget: int = 0, ) -> str:
    """Send a user prompt to an Anthropic model and return the text response."""
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    if thinking_budget > 0:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }
    else:
        params["temperature"] = temperature

    start = time.perf_counter()
    response = client.messages.create(**params)
    latency = time.perf_counter() - start

    # Log some details about the response
    usage = response.usage
    print(f"\tModel: {response.model}")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {usage.input_tokens}")
    print(f"\tOutput tokens: {usage.output_tokens}")
    thinking_tokens = getattr(usage, 'thinking_tokens', None)
    if thinking_tokens is not None:
        print(f"\tThinking tokens: {thinking_tokens}")

    response_text = ""
    for block in response.content:
        if block.type == "text":
            response_text += block.text
    return response_text

## 3. Send user prompt

This section sends and user prompt to the model using the function previously defined.

In [4]:
prompt = "How many tokens is your context window?"

print("=== Basic model  ===")
print("User:", prompt)
response = query_model(prompt)
print("Claude3:", response)
print()
print("=== Advanced model  ===")
print("User:", prompt)
response = query_model(prompt, model="claude-sonnet-4-20250514", thinking_budget=1024)
print("Claude4:", response)

=== Basic model  ===
User: How many tokens is your context window?
	Model: claude-3-haiku-20240307
	Latency: 1.608 seconds
	Input tokens: 15
	Output tokens: 73
Claude3: I do not actually have a fixed context window size. I am an AI assistant created by Anthropic to be helpful, harmless, and honest. I don't have the same architectural details as language models that use a sliding context window. My responses are generated based on my training by Anthropic, not a fixed-size input context.

=== Advanced model  ===
User: How many tokens is your context window?
	Model: claude-sonnet-4-20250514
	Latency: 5.657 seconds
	Input tokens: 44
	Output tokens: 165
Claude4: My context window is approximately 200,000 tokens. This means I can process and maintain context over roughly that amount of text in our conversation, which translates to around 150,000-200,000 words depending on the specific content.

However, I should note that Anthropic hasn't published all the exact technical specifications pub